# Case C — E-commerce recsys
Heterogeneous bipartite graph + two-tower kNN + NeighborLoader minibatching.

In [ ]:
from pathlib import Path
import tempfile, numpy as np, pyarrow as pa
from caracaldb.storage import create_bundle
from caracaldb.storage.node_store import open_node_store
from caracaldb.storage.edge_store import open_edge_store
from caracaldb.graph import build_csr
from caracaldb.graph.csr_reader import CsrReader
from caracaldb.graph.hnsw import HnswConfig, HnswIndex
from caracaldb.ml import NeighborLoader, NeighborLoaderConfig

tmp = Path(tempfile.mkdtemp())
bundle = create_bundle(tmp / 'ec')
users = open_node_store(bundle, class_iri='http://x/User', local_name='User', create=True)
users.append(pa.record_batch({'name': pa.array([f'u{i}' for i in range(4)])}))
viewed = open_edge_store(bundle, property_iri='http://x/viewed', local_name='viewed', create=True)
viewed.append(pa.record_batch({'src': pa.array([0,0,1,2,3], type=pa.uint64()),
                                'dst': pa.array([1,2,0,1,3], type=pa.uint64())}))
csr_path = bundle.path / 'viewed.csr'
build_csr(viewed, num_vertices=4, out_path=csr_path, with_eids=True)
csr = CsrReader(csr_path)

rng = np.random.default_rng(0)
user_emb = rng.standard_normal((4, 8)).astype(np.float32)
item_emb = user_emb.copy()  # match user[0]==item[0]
idx = HnswIndex(HnswConfig(dim=8, max_elements=4))
idx.add(np.arange(4, dtype=np.uint64), item_emb)
labels, dists = idx.search(user_emb[0], k=1)
print('nearest item to user 0 →', labels[0][0], 'distance', float(dists[0][0]))


In [ ]:
cfg = NeighborLoaderConfig(
    layers=[2, 1],
    edge_readers={'http://x/viewed': csr},
    seed_class_iri='http://x/User',
    seed_local_name='User',
    batch_size=2,
)
loader = NeighborLoader(bundle, cfg)
for batch in loader:
    print('Subgraph nodes:', sum(t.num_rows for t in batch.nodes.values()),
          'edges:', sum(t.num_rows for t in batch.edges.values()))
